In [2]:
import numpy as np
a = np.arange(6).reshape(2, 3)
print(a[1, 2], a.shape)

5 (2, 3)


a[1,2] — строка 1, столбец 2 - 5.
a.shape — массив на 2 строки и 3 столбца → (2, 3).
Почему: reshape(2,3) раскладывает 6 чисел (0..5) в таблицу 2×3, и индексация [строка, столбец] считает с нуля

In [3]:
a = np.array([3, 8, 1, 8])
print(a.argmax(), a.max())

1 8


a.argmax() - 1, a.max() - 8.
Почему: максимальное значение 8 встречается дважды — на позициях 1 и 3. .max() просто возвращает значение (8). .argmax() возвращает позицию первого вхождения максимума, то есть индекс 1, а не 3

In [4]:
a = np.arange(10)
b = a[::3]
b[0] = 99
print(a[0])

99


b = a[::3] — это срез с шагом, а любой срез массива NumPy  — это окно в ту же память, а не копия. b[0] соответствует a[0], поэтому изменение через b видно и в a

In [ ]:
a = np.array([1, 5, 7, 2])
print((a > 3).sum(), (a > 3).mean())

маска a > 3 даёт [False, True, True, False]. .sum() считает True как 1, False как 0, итого 2 элемента прошли условие. .mean() — это доля True от всех элементов: 2 из 4 - 0.5

In [5]:
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

(2, 3) 0.25


linspace(начало, конец, количество) делит отрезок на равные интервалы, и конец включается (в отличие от arange). Шаг здесь 1/4 = 0.25, второй элемент (индекс 1) — это 0.25

In [22]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [23]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])


In [24]:
# 1.Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
idx = amount.argmax()
print(amount[idx], cust[idx], city[idx])

9999990.0 TEST Алматы


In [25]:
# 2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
revenue_paid = amount[paid].sum()
revenue_unpaid = amount[~paid].sum()
print(revenue_paid, revenue_unpaid)

101909360.0 7974000.0


In [26]:
# 3 Сколько заказов в Шымкенте и какая у них доля от всех заказов?
shymkent_count = (city == "Шымкент").sum()
shymkent_share = (city == "Шымкент").mean()
print(shymkent_count, shymkent_share)

804 0.1471718835804503


In [27]:
# 4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
# Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
mask_discount = paid & (amount > 50_000)
discount_all = (amount[mask_discount] * 0.15).sum()

mask_no_test = cust != "TEST"
discount_no_test = (amount[mask_discount & mask_no_test] * 0.15).sum()

print(discount_all, discount_no_test)
# Разница в разы: 3 заказа TEST по 9 999 990 ₸ оплачены и дороже 50000,
# поэтому сами по себе дают огромный вклад в "скидку"

5702422.5 1202427.0


In [ ]:
# 5
ok = ~np.isnan(days)
for c in np.unique(city):
    mask_c = (city == c) & ok
    share = (days[mask_c] <= 2).mean()
    print(c, round(share, 3))

In [28]:
# задание 6 найти ошибку
# ошибка 1: в данных есть клиент TEST с суммами по 9 999 990 - это тестовые записи, не реальные заказы
# ошибка 2: есть задублированные строки заказов, из-за них среднее тоже сдвигается

avg = amount[cust != "TEST"].mean()
print("Средний чек:", round(avg))

Средний чек: 14631
